# Notebook 5 — API Server
Runs FastAPI + serves predictions via ngrok (Colab) or directly (local).

**For Colab:** Run as-is. Will expose a public ngrok URL.
**For local:** Run `uvicorn api_server:app --port 8000` in the terminal instead.

Requirements: `tgn_best.pt` and one or more `graph_seed*.pt` files available.
The API accepts ANY uploaded CSV — it builds a fresh initial state from it
and runs the TGN rollout on whichever graph most closely matches the CSV's population size.

In [ ]:
# Cell 1 — Choose environment
import os

ENV = 'colab'   # change to 'local' if running locally

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    BASE        = '/content/drive/MyDrive/epidemic_project'
    MODEL_PATH  = f'{BASE}/model/tgn_best.pt'
    GRAPHS_DIR  = f'{BASE}/graphs'
    DATA_DIR    = f'{BASE}/data'
else:
    # Local — adjust these paths to wherever your files are
    BASE        = os.path.expanduser('~/epidemic_project')
    MODEL_PATH  = f'{BASE}/model/tgn_best.pt'
    GRAPHS_DIR  = f'{BASE}/graphs'
    DATA_DIR    = f'{BASE}/data'

print('Model:', MODEL_PATH)
print('Graphs:', GRAPHS_DIR)
print('Available graphs:', [f for f in os.listdir(GRAPHS_DIR) if f.endswith('.pt')])

In [ ]:
# Cell 2 — Install
!pip install -q fastapi uvicorn pyngrok python-multipart torch-geometric
if ENV == 'colab':
    !pip install -q torch-scatter torch-sparse \
        -f https://data.pyg.org/whl/torch-2.2.0+cu118.html

In [ ]:
# Cell 3 — Write the API to a file
api_code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
import pandas as pd
import numpy as np
import json, io, os, uuid
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import Optional, Dict

MODEL_PATH = os.environ.get("MODEL_PATH", "tgn_best.pt")
GRAPHS_DIR = os.environ.get("GRAPHS_DIR", "graphs")
device     = torch.device("cpu")

ZONE_CENTROIDS = {
    1:(18.5204,73.8567), 2:(18.5314,73.8446), 3:(18.5089,73.8741),
    4:(18.4968,73.8631), 5:(18.5421,73.8789), 6:(18.5612,73.8234),
    7:(18.4823,73.9012), 8:(18.5198,73.9156), 9:(18.4712,73.8345),
    10:(18.5534,73.8678),11:(18.4634,73.8912),12:(18.5023,73.8123),
    13:(18.5756,73.8456),14:(18.4891,73.8234),15:(18.5345,73.9234),
    16:(18.5067,73.7987),17:(18.4756,73.9345),18:(18.5589,73.8012),
    19:(18.4623,73.8678),20:(18.5234,73.8901),
}

# ---- Model definition (must match training) ----
class NodeMemory(nn.Module):
    def __init__(self, mem_dim):
        super().__init__()
        self.mem_dim = mem_dim
        self.memory  = None
    def init(self, N, device):
        self.memory = torch.zeros(N, self.mem_dim, device=device)
    def get(self):       return self.memory
    def update(self, m): self.memory = m.detach()

class EpidemicTGN(nn.Module):
    def __init__(self, node_feat_dim=9, mem_dim=32, state_dim=5):
        super().__init__()
        self.mem_dim = mem_dim; self.state_dim = state_dim
        self.memory  = NodeMemory(mem_dim)
        msg_in       = mem_dim*2+1+state_dim
        self.msg_fn  = nn.Sequential(nn.Linear(msg_in,64),nn.ReLU(),nn.Linear(64,mem_dim))
        self.mem_updater = nn.GRUCell(mem_dim,mem_dim)
        gat_in       = mem_dim+node_feat_dim+state_dim
        self.gat1    = GATv2Conv(gat_in,32,heads=2,concat=False,edge_dim=1,dropout=0.0,add_self_loops=True)
        self.gat2    = GATv2Conv(32,32,heads=2,concat=False,edge_dim=1,dropout=0.0,add_self_loops=True)
        self.norm1   = nn.LayerNorm(32); self.norm2 = nn.LayerNorm(32)
        self.predictor = nn.Sequential(nn.Linear(32,64),nn.LayerNorm(64),nn.ReLU(),nn.Dropout(0.0),nn.Linear(64,state_dim))
    def _step(self, x_state, node_feats, edge_index, edge_attr):
        x_state=x_state.float(); node_feats=node_feats.float(); edge_attr=edge_attr.float()
        N=x_state.shape[0]; src=edge_index[0]; dst=edge_index[1]; mem=self.memory.get().float()
        e_w=edge_attr.squeeze(1) if edge_attr.dim()==2 else edge_attr
        msg_input=torch.cat([mem[src],mem[dst],e_w.unsqueeze(1),x_state[src]],dim=-1)
        msgs=self.msg_fn(msg_input).float()
        agg=torch.zeros(N,self.mem_dim,device=mem.device,dtype=torch.float32)
        idx=dst.unsqueeze(1).expand(dst.shape[0],self.mem_dim)
        agg.scatter_add_(0,idx,msgs)
        new_mem=self.mem_updater(agg,mem); self.memory.update(new_mem)
        e_2d=edge_attr if edge_attr.dim()==2 else edge_attr.unsqueeze(1)
        h=torch.cat([new_mem,node_feats,x_state],dim=-1)
        h=self.norm1(F.elu(self.gat1(h,edge_index,e_2d)))
        h=self.norm2(F.elu(self.gat2(h,edge_index,e_2d)))
        return F.softmax(self.predictor(h),dim=-1)
    @torch.no_grad()
    def rollout(self, x0, node_feats, edge_index, edge_attr, n_days=30):
        self.memory.init(x0.shape[0], x0.device)
        preds=[]; x=x0.float()
        for _ in range(n_days):
            x=self._step(x,node_feats,edge_index,edge_attr); preds.append(x)
        return torch.stack(preds,dim=0)

# ---- Load all available graphs ----
print("Loading graphs...")
graphs = {}
for fname in os.listdir(GRAPHS_DIR):
    if not fname.endswith(".pt"): continue
    seed = int(fname.replace("graph_seed","").replace(".pt",""))
    g    = torch.load(os.path.join(GRAPHS_DIR,fname), map_location=device)
    graphs[seed] = {
        "edge_index": g.edge_index,
        "edge_attr":  g.edge_attr,
        "node_feats": g.x,
        "pos":        g.pos.numpy(),
        "zone":       g.zone.numpy(),
        "N":          g.num_nodes,
    }
    print(f"  Graph seed={seed}: N={g.num_nodes}")

print("Loading model...")
model = EpidemicTGN()
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print("Model loaded.")

# Use the largest available graph by default for inference
default_seed = max(graphs.keys(), key=lambda s: graphs[s]["N"])
print(f"Default inference graph: seed={default_seed}, N={graphs[default_seed]["N"]}")

# In-memory store
store: Dict[str, torch.Tensor] = {}

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

class PredictRequest(BaseModel):
    graph_id:      str
    interventions: Optional[dict] = {}

@app.get("/health")
def health():
    return {"status":"ok","graphs":list(graphs.keys()),"default_seed":default_seed}

@app.post("/api/upload")
async def upload(file: UploadFile = File(...)):
    contents = await file.read()
    try:
        df = pd.read_csv(io.StringIO(contents.decode()))
    except Exception as e:
        raise HTTPException(400, f"CSV parse error: {e}")

    g  = graphs[default_seed]
    N  = g["N"]
    x0 = torch.zeros(N, 5)
    x0[:, 0] = 1.0

    if "node_id" in df.columns and "infected" in df.columns:
        for _, row in df[df["infected"]==1].iterrows():
            nid = int(row["node_id"])
            if nid < N:
                x0[nid, 0] = 0.0; x0[nid, 2] = 1.0

    gid = str(uuid.uuid4())[:8]
    store[gid] = x0
    return {"graph_id":gid, "n_infected":int((x0[:,2]>0).sum()), "n_nodes":N}

@app.post("/api/predict")
def predict(req: PredictRequest):
    if req.graph_id not in store:
        raise HTTPException(404, "graph_id not found")

    g  = graphs[default_seed]
    x0 = store[req.graph_id]
    ea = g["edge_attr"].clone()
    iv = req.interventions or {}
    if iv.get("mask_mandate"):
        ea = ea * (1 - 0.5 * float(iv["mask_mandate"]) / 100.0)
    if iv.get("school_closure"): ea = ea * 0.7
    if iv.get("lockdown"):       ea = ea * 0.4

    # MC dropout for uncertainty
    N_MC = 10; preds = []
    model.train()
    with torch.no_grad():
        for _ in range(N_MC):
            p = model.rollout(x0, g["node_feats"], g["edge_index"], ea, n_days=30)
            preds.append(p.numpy())
    model.eval()

    preds_arr = np.stack(preds, axis=0)
    mean_pred = preds_arr.mean(axis=0)
    std_pred  = preds_arr.std(axis=0)

    node_pos  = g["pos"]
    node_zone = g["zone"]

    nodes_out = []
    for i in range(g["N"]):
        nodes_out.append({
            "id":   i,
            "lat":  round(float(node_pos[i,0]),6),
            "lng":  round(float(node_pos[i,1]),6),
            "zone": int(node_zone[i]),
            "days": [{"S":round(float(mean_pred[t,i,0]),4),
                      "E":round(float(mean_pred[t,i,1]),4),
                      "I":round(float(mean_pred[t,i,2]),4),
                      "R":round(float(mean_pred[t,i,3]),4),
                      "D":round(float(mean_pred[t,i,4]),4),
                      "u":round(float(std_pred[t,i,2]),4)}
                     for t in range(30)]
        })

    zones_out = []
    for z in range(1,21):
        mask = node_zone == z
        if not mask.any(): continue
        zone_I  = mean_pred[:, mask, 2].mean(axis=1)
        clat,clng = ZONE_CENTROIDS.get(z,(0.0,0.0))
        zones_out.append({
            "zone":     z,
            "clat":     clat,
            "clng":     clng,
            "severity": [round(float(v),4) for v in zone_I],
            "peak_day": int(zone_I.argmax())+1,
        })

    return {"nodes": nodes_out, "zones": zones_out}
'''

with open('/content/api_server.py', 'w') as f:
    f.write(api_code)

# Set environment variables for the API
os.environ['MODEL_PATH'] = MODEL_PATH
os.environ['GRAPHS_DIR'] = GRAPHS_DIR
print('api_server.py written')

In [ ]:
# Cell 4 — Start server + ngrok tunnel
import subprocess, threading, time
from pyngrok import ngrok, conf

NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'   # get free token at ngrok.com
conf.get_default().auth_token = NGROK_TOKEN

def run_server():
    subprocess.run(['uvicorn', 'api_server:app',
                    '--host', '0.0.0.0', '--port', '8000'])

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(4)

tunnel  = ngrok.connect(8000, 'http')
API_URL = tunnel.public_url
print(f'\nAPI live at: {API_URL}')
print(f'Health:      {API_URL}/health')
print(f'\nPaste this URL into your frontend .env as VITE_API_URL={API_URL}')

In [ ]:
# Cell 5 — Test the API end-to-end
import requests

# Create a test CSV in memory
test_csv = 'node_id,infected\n0,1\n100,1\n500,1\n1000,1\n2000,1\n'

# Upload
r1 = requests.post(f'{API_URL}/api/upload',
                   files={'file': ('test.csv', test_csv, 'text/csv')})
print('Upload:', r1.json())
gid = r1.json()['graph_id']

# Predict
print('Running prediction (2-5 mins on CPU)...')
import time; t0 = time.time()
r2 = requests.post(f'{API_URL}/api/predict',
                   json={'graph_id': gid, 'interventions': {}},
                   timeout=600)
print(f'Done in {time.time()-t0:.0f}s')
data = r2.json()
print('Keys:', list(data.keys()))
print('Nodes:', len(data['nodes']))
print('Zones:', len(data['zones']))
print('Day 1 node 0:', data['nodes'][0]['days'][0])
print('Day 30 node 0:', data['nodes'][0]['days'][29])